In [1]:
import numpy as np
import plotly.graph_objects as go
from astropy.time import Time
from astropy.coordinates import (
    GCRS, ITRS, EarthLocation,
    CartesianRepresentation
)
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec

print("Imports loaded!")

Imports loaded!


In [2]:
# Fetch latest TLE for ISS
import requests

url = "https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=TLE"
response = requests.get(url)
lines = response.text.strip().split('\n')

name  = lines[0].strip()
line1 = lines[1].strip()
line2 = lines[2].strip()

print(f"TLE fetched for: {name}")
print(f"Line 1: {line1}")
print(f"Line 2: {line2}")

TLE fetched for: ISS (ZARYA)
Line 1: 1 25544U 98067A   26093.87567842  .00012431  00000+0  23534-3 0  9997
Line 2: 2 25544  51.6327 307.8415 0006253 268.9169  91.1103 15.48754008560234


In [3]:
# Parse TLE and extract epoch
satellite = Satrec.twoline2rv(line1, line2)

# Extract epoch
epoch = Time(satellite.jdsatepoch + satellite.jdsatepochF, format='jd', scale='utc')

# Time since TLE was published
now = Time(datetime.now(timezone.utc))
tle_age = (now - epoch).to(u.hour)

print(f"Satellite parsed!")
print(f"   Epoch:         {epoch.iso}")
print(f"   Current time:  {now.iso}")
print(f"   TLE age:       {tle_age:.2f}")

# Warn if TLE is getting stale
if tle_age.value > 48:
    print(f"WARNING: TLE is over 48 hours old — accuracy may be degraded")
elif tle_age.value > 24:
    print(f"TLE is over 24 hours old — consider refreshing")
else:
    print(f"TLE is fresh")

Satellite parsed!
   Epoch:         2026-04-03 21:00:58.615
   Current time:  2026-04-04 05:24:01.091
   TLE age:       8.38 h
TLE is fresh


In [4]:
# Propagate orbit from epoch for one full period
num_steps = 500

# Get orbital period from mean motion (revs per day → seconds per rev)
mean_motion = satellite.no  # radians per minute
period_s = (2 * np.pi / mean_motion) * 60  # convert to seconds

print(f"Orbital period: {period_s/60:.2f} minutes")

time_deltas = np.linspace(0, period_s, num_steps)

lats_epoch, lons_epoch, alts_epoch = [], [], []

for dt in time_deltas:
    # SGP4 propagation
    t = epoch + dt * u.s
    e, r_teme, v_teme = satellite.sgp4(t.jd1, t.jd2)

    # TEME → ITRS → lat/lon
    r_teme_coord = CartesianRepresentation(r_teme * u.km)
    from astropy.coordinates import TEME
    teme = TEME(r_teme_coord, obstime=t)
    itrs = teme.transform_to(ITRS(obstime=t))
    loc = EarthLocation(*itrs.cartesian.xyz)

    lats_epoch.append(loc.lat.deg)
    lons_epoch.append(loc.lon.deg)
    alts_epoch.append(loc.height.to(u.km).value)

lats_epoch = np.array(lats_epoch)
lons_epoch = np.array(lons_epoch)
alts_epoch = np.array(alts_epoch)

print(f" Epoch orbit propagated!")
print(f"   Lat range:  {lats_epoch.min():.1f}° to {lats_epoch.max():.1f}°")
print(f"   Lon range:  {lons_epoch.min():.1f}° to {lons_epoch.max():.1f}°")
print(f"   Alt range:  {alts_epoch.min():.1f} km to {alts_epoch.max():.1f} km")

Orbital period: 92.98 minutes
 Epoch orbit propagated!
   Lat range:  -51.8° to 51.8°
   Lon range:  -179.7° to 179.8°
   Alt range:  421.0 km to 433.9 km


In [5]:
def break_antimeridian(lons):
    lons_plot = lons.copy().astype(float)
    for i in range(1, len(lons_plot)):
        if abs(lons_plot[i] - lons_plot[i-1]) > 180:
            lons_plot[i-1] = None
    return lons_plot

fig_ground = go.Figure()

fig_ground.add_trace(go.Scattergeo(
    lon=break_antimeridian(lons_epoch),
    lat=lats_epoch,
    mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    name='Track from Epoch'
))
fig_ground.add_trace(go.Scattergeo(
    lon=[lons_epoch[0]],
    lat=[lats_epoch[0]],
    mode='markers+text',
    marker=dict(size=8, color='yellow'),
    text=['ISS Epoch'],
    textfont=dict(color='white', size=11),
    name='ISS Epoch'
))
fig_ground.update_layout(
    title='🛰️ ISS Ground Track — Epoch',
    geo=dict(
        showland=True, landcolor='darkgreen',
        showocean=True, oceancolor='midnightblue',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True, countrycolor='gray',
        showframe=False,
        bgcolor='black',
        projection_type='natural earth'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fig_ground.show()
print("Epoch ground track plotted!")

Epoch ground track plotted!


In [6]:
# Compute current ISS position
now = Time(datetime.now(timezone.utc))

# SGP4 propagate from epoch to now
e, r_teme, v_teme = satellite.sgp4(now.jd1, now.jd2)

# TEME → ITRS → lat/lon
from astropy.coordinates import TEME
r_teme_coord = CartesianRepresentation(r_teme * u.km)
teme = TEME(r_teme_coord, obstime=now)
itrs = teme.transform_to(ITRS(obstime=now))
loc = EarthLocation(*itrs.cartesian.xyz)

iss_lat_now = loc.lat.deg
iss_lon_now = loc.lon.deg
iss_alt_now = loc.height.to(u.km).value

print(f"   Current ISS position:")
print(f"   Latitude:  {iss_lat_now:.2f}°")
print(f"   Longitude: {iss_lon_now:.2f}°")
print(f"   Altitude:  {iss_alt_now:.2f} km")
print(f"   As of:     {now.iso} UTC")

   Current ISS position:
   Latitude:  23.79°
   Longitude: -167.75°
   Altitude:  423.21 km
   As of:     2026-04-04 05:24:03.506 UTC


In [7]:
# Propagate one full orbit from current position
time_deltas = np.linspace(0, period_s, num_steps)

lats_now, lons_now, alts_now = [], [], []

for dt in time_deltas:
    t = now + dt * u.s
    e, r_teme, v_teme = satellite.sgp4(t.jd1, t.jd2)

    r_teme_coord = CartesianRepresentation(r_teme * u.km)
    teme = TEME(r_teme_coord, obstime=t)
    itrs = teme.transform_to(ITRS(obstime=t))
    loc = EarthLocation(*itrs.cartesian.xyz)

    lats_now.append(loc.lat.deg)
    lons_now.append(loc.lon.deg)
    alts_now.append(loc.height.to(u.km).value)

lats_now = np.array(lats_now)
lons_now = np.array(lons_now)
alts_now = np.array(alts_now)

print(f"   Current orbit propagated!")
print(f"   Start: {lats_now[0]:.2f}°, {lons_now[0]:.2f}°")
print(f"   Should match current position:")
print(f"   ISS now: {iss_lat_now:.2f}°, {iss_lon_now:.2f}°")

   Current orbit propagated!
   Start: 23.79°, -167.75°
   Should match current position:
   ISS now: 23.79°, -167.75°


In [8]:
fig_ground2 = go.Figure()

# ── Epoch track (past) ────────────────────────────────────────────
fig_ground2.add_trace(go.Scattergeo(
    lon=break_antimeridian(lons_epoch),
    lat=lats_epoch,
    mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    name='Track from Epoch (past)'
))

# ── Current + future track ────────────────────────────────────────
fig_ground2.add_trace(go.Scattergeo(
    lon=break_antimeridian(lons_now),
    lat=lats_now,
    mode='lines',
    line=dict(color='cyan', width=1.5),
    name='Track from Now (future)'
))

# ── ISS epoch position ────────────────────────────────────────────
fig_ground2.add_trace(go.Scattergeo(
    lon=[lons_epoch[0]],
    lat=[lats_epoch[0]],
    mode='markers+text',
    marker=dict(size=8, color='yellow'),
    text=['ISS @ Epoch'],
    textfont=dict(color='white', size=11),
    name='ISS @ Epoch'
))

# ── ISS current position ──────────────────────────────────────────
fig_ground2.add_trace(go.Scattergeo(
    lon=[iss_lon_now],
    lat=[iss_lat_now],
    mode='markers+text',
    marker=dict(size=8, color='red'),
    text=['ISS Now'],
    textfont=dict(color='white', size=11),
    name='ISS Now'
))

fig_ground2.update_layout(
    title='🛰️ ISS Ground Track — Epoch vs Now',
    geo=dict(
        showland=True, landcolor='darkgreen',
        showocean=True, oceancolor='midnightblue',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True, countrycolor='gray',
        showframe=False,
        bgcolor='black',
        projection_type='natural earth'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fig_ground2.show()
print(" Done!")

 Done!
